In [2]:
import sys
!{sys.executable} -m pip install ipython-autotime
!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install torch
!{sys.executable} -m pip install optuna
!{sys.executable} -m pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 1.6/101.7 MB 9.4 MB/s eta 0:00:11
   - -------------------------------------- 4.2/101.7 MB 11.0 MB/s eta 0:00:09
   -- ------------------------------------- 6.6/101.7 MB 11.2 MB/s eta 0:00:09
   --- ------------------------------------ 8.9/101.7 MB 11.5 MB/s eta 0:00:09
   ---- ----------------------------------- 11.5/101.7 MB 11.5 MB/s eta 0:00:08
   ----- ---------------------------------- 13.9/101.7 MB 11.6 MB/s eta 0:00:08
   ------ --------------------------------- 16.3/101.7 MB 11.6 MB/s eta 0:00:08
   ------- -------------------------------- 18.4/101.7 MB 11.5 MB/s eta 0:00:08
   -------- ------------------------------- 21.0/101.7 MB 11.4 MB/s eta 0:00:08
   --------- ------------------------------ 23.3/101.7 MB 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable


In [1]:
%load_ext autotime

import os
import random
import gc
from itertools import combinations
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, log_loss
import joblib
import zipfile
import ast

time: 3.52 s (started: 2026-05-25 22:09:38 +09:00)


### File Setting

In [9]:
SEEDS = [42, 106, 1031]
RATIOS = [10]

def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

time: 0 ns (started: 2026-05-25 03:32:07 +09:00)


### File Load

In [2]:
base_path = "C:\\Users\\woolz\\Downloads\\open\\"
file_names = [
    "experiment_sample_xgb_42_10.parquet",
    "experiment_sample_xgb_106_10.parquet",
    "experiment_sample_xgb_1031_10.parquet"
]

datasets = {}
for i, fname in enumerate(file_names, start=1):
    df = pd.read_parquet(base_path + fname)
    key = f"train_{i:02d}" # 2자리의 정수 형식
    datasets[key] = df
    print(f"[Loaded] {key} | shape={df.shape}")

train_01 = datasets["train_01"]
train_02 = datasets["train_02"]
train_03 = datasets["train_03"]

[Loaded] train_01 | shape=(227084, 119)
[Loaded] train_02 | shape=(227084, 119)
[Loaded] train_03 | shape=(227084, 119)
time: 3.57 s (started: 2026-05-25 22:09:48 +09:00)


### Data Processing

In [3]:
def create_combination_features(df):

    base_cols = ['gender', 'age_group', 'inventory_id', 'day_of_week', 'hour']
    combo_features = {}

    for col_a, col_b in combinations(base_cols, 2): # combinations: base_cols 중 2개씩 뽑기
        combo_name = f'{col_a}_{col_b}'
        combo_features[combo_name] = (df[col_a].astype(str) + '_' + df[col_b].astype(str)).astype(object)

    combo_df = pd.DataFrame(combo_features, index=df.index) # index=df.index: 기존 df와 인덱스를 동일하게 맞추기
    df = pd.concat([df, combo_df], axis=1)

    print(f"✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})") # 총 10개의 feature 생성
    return df

train_01 = create_combination_features(train_01)
train_02 = create_combination_features(train_02)
train_03 = create_combination_features(train_03)

✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 129)
✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 129)
✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 129)
time: 1.11 s (started: 2026-05-25 22:09:56 +09:00)


In [4]:
df_click_prob = pd.read_excel('high_click_numbers.xlsx')

# zip으로 (428, 0.17) -> dict은 (key, value) 형태를 기대하므로 {428: 0.17}로 변환
click_prob_map = dict(zip(df_click_prob['number'], df_click_prob['click_prob']))

pos_list = {370, 528, 68, 561, 144, 227, 417, 442, 186, 395}
neg_list = {154, 222, 84, 498, 434, 511, 216, 497, 309, 446}

def add_seq_features(df, name="dataset"):
    seq_len, avg_prob, seq_neg, seq_pos = [], [], [], []

    for s in df["seq"]:
        if isinstance(s, str) and s != "": # 샘플 별 seq가 문자열인지, 빈 문자열은 아닌지 확인
            arr = [int(x) for x in s.split(",") if x]

            seq_len.append(len(arr))

            probs = [click_prob_map[num] for num in arr if num in click_prob_map] # 각 숫자에 대한 확률 lookup
            avg_prob.append(sum(probs) / len(probs) if probs else np.nan) # probs 없으면 결측값 처리

            seq_neg.append(sum(1 for x in arr if x in neg_list))
            seq_pos.append(sum(1 for x in arr if x in pos_list))
        else:
            seq_len.append(0)
            avg_prob.append(np.nan)
            seq_neg.append(0)
            seq_pos.append(0)

    df["seq_len"] = seq_len
    df["avg_click_prob"] = avg_prob
    df["seq_neglogcount"] = seq_neg
    df["seq_poslogcount"] = seq_pos

    print(f"✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})")
    return df

train_01 = add_seq_features(train_01)
train_02 = add_seq_features(train_02)
train_03 = add_seq_features(train_03)

✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: 133)
✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: 133)
✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: 133)
time: 1min 59s (started: 2026-05-25 22:10:03 +09:00)


In [5]:
for df in [train_01, train_02, train_03]:
    history_b_cols = [
        c for c in df.columns
        if c.startswith('history_b_')
    ]

    df['history_b_mean'] = (df[history_b_cols].mean(axis=1))
    df['history_b_std'] = (df[history_b_cols].std(axis=1, ddof=0)) # ddof=1(default)은 통계적 추정 시 사용, engineering에선 0 사용
    df['history_b_max'] = (df[history_b_cols].max(axis=1))
    df['history_b_missing'] = (df[history_b_cols].isnull().any(axis=1).astype(int))

    print("Train shape:", df.shape)

Train shape: (227084, 137)
Train shape: (227084, 137)
Train shape: (227084, 137)
time: 515 ms (started: 2026-05-25 22:12:09 +09:00)


In [6]:
full_train_df = pd.concat([train_01, train_02, train_03], ignore_index=True) # 기존 인덱스 버리고 다시 0부터 정렬

agg_targets = ['history_a_1','history_a_2','history_a_3', 'history_a_6','feat_d_4','l_feat_1','l_feat_2']

agg_stats = (
    full_train_df.groupby('inventory_id')[agg_targets]
    .agg(['mean','std'])
    .reset_index() # groupby 결과의 index(inventory_id)를 일반 column으로 변환
)

agg_stats.columns = ['inventory_id'] + [
    f'inventory_id_{col}_{stat}' for col, stat in agg_stats.columns[1:]
]

count_cols = ['age_group_inventory_id', 'inventory_id', 'inventory_id_hour']
count_maps = {col: full_train_df[col].value_counts().to_dict() for col in count_cols}

del full_train_df
gc.collect()


def add_features(df: pd.DataFrame, name: str = "dataset") -> pd.DataFrame:
    """groupby 통계량 + count encoding + 결측치 처리"""
    print(f"\n🚀 {name} 처리 시작...")

    df = df.merge(agg_stats, on='inventory_id', how='left')
    df.fillna(0, inplace=True)  # 숫자형 결측치 처리 (inplace=True: 원본 df 직접 수정)

    for col, cmap in count_maps.items():
        df[f"{col}_count"] = df[col].astype(str).map(cmap).fillna(0).astype(int)

    obj_cols = df.select_dtypes(include='object').columns
    df[obj_cols] = df[obj_cols].fillna("missing").astype(str)

    print(f"✅ {name}: seq 기반 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})")
    return df

train_01 = add_features(train_01)
train_02 = add_features(train_02)
train_03 = add_features(train_03)


🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 154)

🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 154)

🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 154)
time: 4.01 s (started: 2026-05-25 22:12:26 +09:00)


### Modeling

In [7]:
# 클래스 균형 가중치 ~ 데이터 샘플 단위에 적용
def make_class_balanced_weights(y_true):
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()

    w_pos = 0.5 / (n_pos if n_pos > 0 else 1)
    w_neg = 0.5 / (n_neg if n_neg > 0 else 1)
    
    return np.where(y_true == 1, w_pos, w_neg)


# 전처리
BASE_EXCLUDE = {"clicked", "ID", "seq"}
def preprocess(df, exclude_features=None):
    exclude = BASE_EXCLUDE.copy()
    if exclude_features:
        exclude |= set(exclude_features) # |=: set union 연산
    X = df.drop(columns=[c for c in exclude if c in df.columns], errors="ignore")

    for col in X.select_dtypes(include="object").columns:
        X[col] = X[col].astype("category").cat.codes.astype("int32")
    for col in X.select_dtypes(include=["int64"]).columns:
        X[col] = X[col].astype("int32")
    for col in X.select_dtypes(include=["float64"]).columns:
        X[col] = X[col].astype("float32")
    return X


# Optuna 튜닝 함수
def tune_xgb_with_optuna(X, y, seed, n_trials=50): # 50번의 hyperparameter 조합 시도
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)

    scale_pos_weight = float((y_train == 0).sum() / max(1, (y_train == 1).sum())) # float(n_neg / n_pos)

    sampler = TPESampler(seed=seed) 
    study = optuna.create_study(direction="maximize", sampler=sampler)

    def objective(trial):
        params = {
            "objective": "binary:logistic", # 이진 분류
            "eval_metric": "logloss", # validation metric
            "tree_method": "hist",
            "device" : 'cuda',# gpu 기반
            "scale_pos_weight": scale_pos_weight, # imbalance correction 적용
            "eta": trial.suggest_float("eta", 0.01, 0.2, log=True), # learning rate
            "max_depth": trial.suggest_int("max_depth", 4, 12), # tree 깊이
            "subsample": trial.suggest_float("subsample", 0.6, 1.0), # 각 tree 학습 시 일부 row만 사용
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0), # tree마다 일부 feature만 사용
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 200.0), # leaf split 최소 조건
            "gamma": trial.suggest_float("gamma", 0.0, 5.0), # split minimum gain
            "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True), # L2 regularization
            "alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True), # L1 regularization
        }

        booster = xgb.train(
            params, dtrain,
            num_boost_round=1000, # 최대 1000개 tree 생성 가능
            evals=[(dval, "valid")], # 매 boosting round마다 dval 성능도 계산
            early_stopping_rounds=30, # 계속 tree를 추가하다가 validation metric 개선이 30 rounds 동안 없으면 tree 추가 중단
            verbose_eval=False
        )

        p_valid = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1)) # early stopping 이후 tree 사용 x
        ap = average_precision_score(y_val, p_valid)
        wll = log_loss(y_val, p_valid, sample_weight=make_class_balanced_weights(y_val), labels=[0, 1])
        final_score = 0.5 * ap + 0.5 * (1.0 / (1.0 + wll)) # WLL 50%, AP 50%의 score 비중
        return final_score # maximize 대상

    # Optuna 최적화
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params
    best_params.update({
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "device" : 'cuda',
        "scale_pos_weight": float((y == 0).sum() / max(1, (y == 1).sum()))
    })
    return best_params


# 학습+예측 함수
def train_and_predict(train_df, id_tag, seed, n_trials=50):
    print(f"\n=== START PIPELINE: {id_tag} | seed={seed} ===")
    
    random.seed(seed)
    np.random.seed(seed)

    y = train_df["clicked"].astype(int).values.ravel() # target(label) 생성
    X = preprocess(train_df)

    # Optuna 탐색
    print(f"[{id_tag}] Optuna tuning ...")
    best_params = tune_xgb_with_optuna(X, y, seed=seed, n_trials=n_trials)
    print(f"[{id_tag}] Best params:", best_params)

    # 최종 학습
    # HPO 함수 내부의 split은 trial evaluation용이므로, 최종 모델은 별도로 다시 학습 필요 (HPO 때 사용한 split과 동일한 방식으로 다시 split)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    print(f"[{id_tag}] Training final model ...")
    model = xgb.train(
        best_params,
        dtrain,
        num_boost_round=1000,
        evals=[(dtrain, "train"), (dval, "valid")], # 매 boosting round마다 train metric, validation metric 둘 다 추적 
        early_stopping_rounds=50, # validation 성능 개선이 50 boosting rounds동안 없으면 stop
        verbose_eval=100 # 100 rounds마다 metric 출력
    )
    
    # 모델 저장
    model_path = os.path.join('C:\\Users\\woolz\\Downloads\\open\\', f"xgb_{seed}_exp_2.pkl")
    joblib.dump(model, model_path)
    print(f"[{id_tag}] Model saved -> {model_path} | best_iter={model.best_iteration}") # best_iteration = 137 -> 137번째 tree까지가 최고 성능

    print(f"=== END PIPELINE: {id_tag} ===\n")
    return model_path


# 실행
datasets = [
    {"df": train_01, "id": "train_01", "seed": 42},
    {"df": train_02, "id": "train_02", "seed": 106},
    {"df": train_03, "id": "train_03", "seed": 1031}
]

results = {}
for data in datasets:
    model_path = train_and_predict(
        train_df=data["df"],
        id_tag=data["id"],
        seed=data["seed"],
        n_trials=50
    )
    results[data["id"]] = {"model": model_path}

print("=== ALL DONE ===")
print(results)


=== START PIPELINE: train_01 | seed=42 ===
[train_01] Optuna tuning ...


[I 2026-05-25 22:12:47,595] A new study created in memory with name: no-name-dbea49ad-6193-4cf1-98fa-d240135da608
[I 2026-05-25 22:13:07,992] Trial 0 finished with value: 0.39621523606317927 and parameters: {'eta': 0.030710573677773714, 'max_depth': 12, 'subsample': 0.892797576724562, 'colsample_bytree': 0.8394633936788146, 'min_child_weight': 32.04770944804487, 'gamma': 0.7799726016810132, 'lambda': 3.3323645788192616e-08, 'alpha': 0.6245760287469893}. Best is trial 0 with value: 0.39621523606317927.
[I 2026-05-25 22:13:17,923] Trial 1 finished with value: 0.41121971057839884 and parameters: {'eta': 0.06054365855469246, 'max_depth': 10, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978, 'min_child_weight': 166.65608551928392, 'gamma': 1.0616955533913808, 'lambda': 4.329370014459266e-07, 'alpha': 4.4734294104626844e-07}. Best is trial 1 with value: 0.41121971057839884.
[I 2026-05-25 22:13:25,381] Trial 2 finished with value: 0.4349159545394484 and parameters: {'eta

[train_01] Best params: {'eta': 0.01946534863309718, 'max_depth': 6, 'subsample': 0.8939392604269005, 'colsample_bytree': 0.6290890092075, 'min_child_weight': 134.17419167106473, 'gamma': 2.8973120624838744, 'lambda': 9.868745464853728, 'alpha': 1.2403984150261029e-08, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_01] Training final model ...
[0]	train-logloss:0.69061	valid-logloss:0.69070
[100]	train-logloss:0.60220	valid-logloss:0.60769
[200]	train-logloss:0.58409	valid-logloss:0.59311
[300]	train-logloss:0.57323	valid-logloss:0.58516
[400]	train-logloss:0.56571	valid-logloss:0.58000
[500]	train-logloss:0.55908	valid-logloss:0.57548
[600]	train-logloss:0.55365	valid-logloss:0.57194
[700]	train-logloss:0.54779	valid-logloss:0.56796
[800]	train-logloss:0.54263	valid-logloss:0.56467
[900]	train-logloss:0.53688	valid-logloss:0.56101
[999]	train-logloss:0.53190	valid-logloss:0.55779
[train_01] Model save

[I 2026-05-25 22:18:35,828] A new study created in memory with name: no-name-26d70749-73ea-4dfc-81cd-f74782252e30
[I 2026-05-25 22:18:50,984] Trial 0 finished with value: 0.4424596770911775 and parameters: {'eta': 0.010259898367843918, 'max_depth': 12, 'subsample': 0.7798260137416737, 'colsample_bytree': 0.8440875086908464, 'min_child_weight': 170.55990367758548, 'gamma': 3.6940510569846796, 'lambda': 5.886868084847547, 'alpha': 0.5213830060758549}. Best is trial 0 with value: 0.4424596770911775.
[I 2026-05-25 22:19:00,727] Trial 1 finished with value: 0.38107955516558106 and parameters: {'eta': 0.12969245549678007, 'max_depth': 11, 'subsample': 0.7409115664353482, 'colsample_bytree': 0.6848127138204806, 'min_child_weight': 156.30843112731282, 'gamma': 4.4970562619261045, 'lambda': 2.6894091077470376e-08, 'alpha': 1.725942108719862e-07}. Best is trial 0 with value: 0.4424596770911775.
[I 2026-05-25 22:19:04,752] Trial 2 finished with value: 0.44254601035980257 and parameters: {'eta': 0

[train_02] Best params: {'eta': 0.012075905955453702, 'max_depth': 6, 'subsample': 0.8269588025920667, 'colsample_bytree': 0.8350710046367448, 'min_child_weight': 38.322293421420945, 'gamma': 2.4130446556400154, 'lambda': 2.52270334456217e-06, 'alpha': 4.1921670376660846e-07, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_02] Training final model ...
[0]	train-logloss:0.69145	valid-logloss:0.69151
[100]	train-logloss:0.61598	valid-logloss:0.61997
[200]	train-logloss:0.59532	valid-logloss:0.60185
[300]	train-logloss:0.58324	valid-logloss:0.59195
[400]	train-logloss:0.57457	valid-logloss:0.58537
[500]	train-logloss:0.56786	valid-logloss:0.58074
[600]	train-logloss:0.56163	valid-logloss:0.57638
[700]	train-logloss:0.55583	valid-logloss:0.57245
[800]	train-logloss:0.55054	valid-logloss:0.56900
[900]	train-logloss:0.54512	valid-logloss:0.56539
[999]	train-logloss:0.53965	valid-logloss:0.56177
[train_02] Mod

[I 2026-05-25 22:24:39,977] A new study created in memory with name: no-name-75f9b84c-552b-4096-8452-4f09772e35d7
[I 2026-05-25 22:24:45,810] Trial 0 finished with value: 0.4424985389610071 and parameters: {'eta': 0.010030542235435747, 'max_depth': 6, 'subsample': 0.7299275357796907, 'colsample_bytree': 0.7672919056877435, 'min_child_weight': 154.42315013332836, 'gamma': 4.788695606033571, 'lambda': 5.838643100318619e-07, 'alpha': 0.0010984897811838177}. Best is trial 0 with value: 0.4424985389610071.
[I 2026-05-25 22:24:54,131] Trial 1 finished with value: 0.3626273552855752 and parameters: {'eta': 0.11978434297771987, 'max_depth': 10, 'subsample': 0.8028360964022752, 'colsample_bytree': 0.9033177780841783, 'min_child_weight': 17.555385408203744, 'gamma': 3.2365936437527756, 'lambda': 3.0923631972991695e-05, 'alpha': 0.1762792686327148}. Best is trial 0 with value: 0.4424985389610071.
[I 2026-05-25 22:24:58,587] Trial 2 finished with value: 0.4284680935575961 and parameters: {'eta': 0

[train_03] Best params: {'eta': 0.010752378917477863, 'max_depth': 8, 'subsample': 0.8320652919068499, 'colsample_bytree': 0.6574456406901088, 'min_child_weight': 159.37795242818376, 'gamma': 3.380146194766012, 'lambda': 9.295454887549505, 'alpha': 0.28651392138944215, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_03] Training final model ...
[0]	train-logloss:0.69151	valid-logloss:0.69161
[100]	train-logloss:0.60723	valid-logloss:0.61309
[200]	train-logloss:0.58059	valid-logloss:0.59088
[300]	train-logloss:0.56665	valid-logloss:0.58042
[400]	train-logloss:0.55682	valid-logloss:0.57341
[500]	train-logloss:0.54940	valid-logloss:0.56826
[600]	train-logloss:0.54383	valid-logloss:0.56457
[700]	train-logloss:0.53848	valid-logloss:0.56102
[800]	train-logloss:0.53287	valid-logloss:0.55737
[900]	train-logloss:0.52736	valid-logloss:0.55376
[999]	train-logloss:0.52235	valid-logloss:0.55054
[train_03] Model save